In [1]:
# Import packages
import numpy as np
import pandas as pd
import hdbscan
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import datasets, linear_model, metrics
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Load data
df = pd.read_parquet('data_exjobb_070425.parquet')
df.head()

,TransactionId,SalePrice,Lat,Lon,BuildingAge,UtilityArea,LotArea,QualityScore,CloseToBeach,EnergyPerformance,...,MunicipalityMoveInFromCountyFrac,MunicipalityMoveOutToCountyFrac,MunicipalityMoveInFrac2yrChangeMean,MunicipalityMoveOutFrac2yrChangeMean,MunicipalityMoveInFromCountyFrac2yrChangeMean,MunicipalityMoveOutToCountyFrac2yrChangeMean,MunicipalityPopulationMeanDeviation,MedianRentMunicipality,MedianRentMunicipality2yrFracChangeMean,MedianRentMunicipalityMeanDeviation
0,10006194,3200000,56.742153,16.291778,48,143,480,36,0,60.0,...,0.025551,0.021314,0.001816,0.004082,0.000930,0.003437,4.373830,1155.0,0.091030,1.123790
1,10006232,1700000,57.278909,13.646019,54,120,750,32,0,114.0,...,0.010123,0.013869,-0.007777,0.006713,-0.001110,0.002518,1.843030,999.0,0.013451,0.972005
2,10006302,3500000,58.328009,15.103299,47,169,592,30,0,NaN,...,0.034871,0.028112,-0.003267,0.003278,-0.000932,0.000765,1.738860,1046.0,0.075010,1.017735
3,10006371,11200000,57.739437,14.122176,128,250,3311,30,0,88.0,...,0.013410,0.012764,-0.001062,0.001830,0.000457,0.000039,8.857676,1059.0,0.061507,1.030384
4,10006399,591000,57.784585,16.146614,112,87,3406,28,0,101.0,...,0.006766,0.006466,-0.001024,-0.000181,0.000441,0.000045,2.279611,909.0,0.009151,0.884437


In [2]:
# Create functions

def remove_non_numbers(data):
    """Remove colums with data which are not numbers and return the matrix, without the columns, and the indexes of the removed rows."""
    del_data = data.copy()
    del_idx = {}

    # Removing all non int or float type features looping from the end.
    iterate = len(del_data[0,:])-1
    for i in range(iterate+1):
        if type(del_data[0,iterate-i]) != float and type(del_data[0,iterate-i]) != int:
            # print(iterate-i, type(del_data[0,iterate-i]), data[0,iterate-i])
            del_data = np.delete(del_data, obj=iterate-i, axis=1)
            del_idx[iterate-i] = iterate-i
    """
    for i in range(len(del_data[0,:])):
        print(i, type(del_data[0,i]), del_data[0,i])

    for i in range(len(data[0,:])):
    print(i, type(data[0,i]), data[0,i])
        """
    return del_data, del_idx


def normalize_feature(feature):
    """Min-max normalization."""
    # This retains the relative position of the data points.
    return (feature-np.min(feature)) / (np.max(feature)-np.min(feature))

def normalize(data, axis=1):
    """Using the normalize_feature function on an np.array along a chosen axis."""
    norm_data = np.zeros_like(data)
    for i in range(data.shape[axis]):
        norm_data[:,i] = normalize_feature(data[:,i])
    return norm_data

def clustering(data, cluster_size=10, min_samples=10, distance_type='euclidean'):
    """Takes data, minimum cluster size, distance metric and returns an array of the data with an added column at index 0 with the cluster label and the number of clusters."""
    clusterer = hdbscan.HDBSCAN(min_cluster_size=cluster_size, min_samples=min_samples ,  metric=distance_type)    # , approx_min_span_tree=True, algorithm='boruvka_balltree'
    clusterer.fit(data)
    return clusterer.labels_.max()+1 , np.insert(data, 0, clusterer.labels_.flatten(), axis=1)

def center_of_mass(number_of_clusters, clusters):
    """Takes a number of clusters and a matrix with points declaring where the first column references which cluster to which the point belongs and
    the rest the coordinates of the point."""
    cluster_centers = np.zeros((number_of_clusters, len(clusters[0])))
    balancing_vector = np.zeros_like(cluster_centers)
    clusters[:,0] += 1
    for i in range(len(clusters[:,None])):
        if clusters[i,0] > 0:
            cluster_centers[int(clusters[i,0])-1] += clusters[i]
            balancing_vector[int(clusters[i,0])-1] += 1 # Keeps track of how many points are in each cluster.
    return cluster_centers / balancing_vector




def reduce_dimension(vector,clusters):
    """Takes a vector and a set of clusters and reduces the vectors dimension to the number of clusters using the clusters as a basis."""
    # Vector projection
    transformation_matrix = np.zeros([len(clusters), len(vector)])

    for i in clusters:
        transformation_matrix[i] = center_of_mass(clusters[i])

    reduced_vector = transformation_matrix * vector
    return reduced_vector

def get_cmap(n, name='hsv'):
    '''Returns a function that maps each index in 0, 1, ..., n-1 to a distinct 
    RGB color; the keyword argument name must be a standard mpl colormap name.'''
    return plt.cm.get_cmap(name, n)

def show_clusters(sorted_clusters, number_of_clusters, centers_of_mass, clusterspan):
    """Creates a 3D plot of the clusters and their center of mass."""
    
    # %matplotlib widget

    fig = plt.figure(figsize = (10,10))
    ax = plt.axes(projection='3d')
    ax.grid()

    cmap = get_cmap((number_of_clusters+1)*2)

    for i in range(1,number_of_clusters+1):
        x1 = sorted_clusters[int(clusterspan[i]):int(clusterspan[i+1]), 1]
        y1 = sorted_clusters[int(clusterspan[i]):int(clusterspan[i+1]), 2]
        z1 = sorted_clusters[int(clusterspan[i]):int(clusterspan[i+1]), 3]
        ax.scatter(x1, y1, z1, c = cmap(i), s = 100)
        xc1,yc1,zc1 = centers_of_mass[i-1, 1:4]
        print(i, xc1, yc1, zc1)
        ax.scatter(xc1,yc1,zc1, c = cmap(-i), s = 100)

    # xc2,yc2,zc2 = centers_of_mass[1, 1:4]

    # print("1", xc1, yc1, zc1)
    # print("2", xc2, yc2, zc2)

    ax.set_title('3D Scatter Plot')

    # Set axes label
    ax.set_xlabel('x', labelpad=20)
    ax.set_ylabel('y', labelpad=20)
    ax.set_zlabel('z', labelpad=20)

def compare_Euclidean(reference, comparables, k=3):
    """Takes a reference point and a matrix of comparable points and returns the first column of the comparable points (ID) and
    the Euclidean distances sorted by distance from low to high."""
    # abs_compare = np.abs(comparables[:,1:] - reference[1:])
    sq_compare = np.square(comparables[:,1:] - reference[1:])
    # print("sq_compare", sq_compare)
    sum_compare = np.sum(sq_compare, axis=1)
    # print("sum_compare", sum_compare)
    Euc_compare = np.sqrt(sum_compare)
    # print("Euc_compare", Euc_compare)

    # np.insert(Euc_compare, 0, comparables[:, 0].flatten())  # Insert IDs back
    # print(comparables[:, 0], Euc_compare)
    Euc_compare = np.vstack((comparables[:, 0], Euc_compare)).T
    # print("Euc_compare", Euc_compare)

    sorted_dis = Euc_compare[Euc_compare[:, 1].argsort()]   # Sort Euc_compare

    return sorted_dis[0:k, :]

def show_clusters_in_latlon(data, sorted_clusters, number_of_clusters, clusterspan, lat_index=2, lon_index=3):
    """Creates a 3D plot of the clusters and their center of mass."""
    
    # %matplotlib widget

    fig = plt.figure(figsize = (10,10))
    ax = plt.axes()
    ax.grid()

    cmap = get_cmap((number_of_clusters+1)*2)

    for i in range(1,number_of_clusters+1):
        x1 = sorted_clusters[int(clusterspan[i]):int(clusterspan[i+1]), lat_index]
        y1 = sorted_clusters[int(clusterspan[i]):int(clusterspan[i+1]), lon_index]
        # z1 = sorted_clusters[int(clusterspan[i]):int(clusterspan[i+1]), 3]
        ax.scatter(x1, y1, c = cmap(i), s = 1)
        # xc1,yc1,zc1 = centers_of_mass[i-1, 1:4]
        # print(i, xc1, yc1, zc1)
        # ax.scatter(xc1,yc1,zc1, c = cmap(-i), s = 100)

    # xc2,yc2,zc2 = centers_of_mass[1, 1:4]

    # print("1", xc1, yc1, zc1)
    # print("2", xc2, yc2, zc2)

    ax.set_title('Scatter Plot')

    # Set axes label
    ax.set_xlabel('x', labelpad=20)
    ax.set_ylabel('y', labelpad=20)
    # ax.set_zlabel('z', labelpad=20)
    

# Outline
1. Find Clusters using hdbscan.
2. Find the centers of the clusters.
3. Reduce the dimensionality to the number of clusters and create a new vector space.
4. Find a suitable model for finding comparisons.

## TODO
1. Linear regression.
2. Neural network.
3. Prepare data (choose features)

# Change to np.array and choose features

Start with excluding the feature that are not int or float.
Later give the unused features numerical values and implement them as well.


In [3]:
# Chaning to numpy array
df = df.fillna(0)
data = df.to_numpy()

data, removed_indices = remove_non_numbers(data)
print("Removed indices:", removed_indices)



data = data.astype(float) # Changing from type object to float so that numpy functions work properly.

data_original = data.copy()

# Divide the data set into training and testing data
percent = 0.10  # Fraction used for testing
samples = int(len(data[:,None]) * percent)

training_data, testing_data = train_test_split(data, test_size=samples)

print('X1 shape: ', training_data.shape)
print('X2 shape: ', testing_data.shape)


data_points = 200
play_data = data[0:data_points, 1:4] # Only using a part of the data.

data = play_data


Removed indices: {153: 153, 49: 49, 48: 48, 40: 40, 39: 39, 32: 32, 26: 26, 25: 25, 24: 24, 23: 23, 17: 17, 15: 15, 10: 10, 0: 0}
X1 shape:  (82467, 160)
X2 shape:  (9163, 160)


# Clustering

In [4]:
def create_clusterspan(sorted_clusters, number_of_clusters):
    clusterspan = np.zeros(number_of_clusters + 2)

    j = 0
    for i in range(len(sorted_clusters[:,0])-1):
        if sorted_clusters[i, 0] < sorted_clusters[i+1,0]:
            # print(sorted_clusters[i, 0])
            clusterspan[j+1] = i+1
            j += 1
    clusterspan[-1] = len(sorted_clusters[:,0])
    return clusterspan

# Normalizing the data
norm_data = normalize(data)

# Finding center-of-mass

In [5]:
# Improved Silhouette method



def Silhouette(SC, NoC, CoM, CS):
    # SC = sorted clusters, NoC = number of clusters, CoM = centers of mass, CS = clusterspan
    

    average_silhouette_clusters = np.zeros((NoC, 2))

    # Find nearest cluster
    # CS = np.delete(CS, 0)
    for i in range(1, NoC+1):
        # print(i)
        average_silhouette_clusters[i-1, :] = i, cluster_silhouette(SC, CS, i, CoM)
        
    # print(average_silhouette_clusters)

    
    average_overall_silhouette = np.sum(average_silhouette_clusters[:, 1]) / NoC
    return average_overall_silhouette, average_silhouette_clusters

In [41]:
def calculate_silhouette(data, start, end, step_size):
    max_iterations = (end-start) // step_size + 1
    Sil_mat = np.zeros((max_iterations, 3))
    # Sil_mat = np.zeros((data_points-2, 2))

    for MCS in range(start, end+1, step_size): # range(2, data_points): # MCS = minimum cluster size  range(9, 11): # 
        index = (MCS-start)//step_size
        print('MCS:', MCS)
        number_of_clusters, clustered_data = clustering(data, cluster_size=MCS, min_samples=2)
        print(number_of_clusters)
        
        if number_of_clusters < 2:
            Sil_mat[index] = MCS, 0, number_of_clusters
            continue
        sorted_clusters = clustered_data[clustered_data[:, 0].argsort()] # sorting the clustered data by cluster affiliation
        sorted_clusters[:,0] +=1

        clusterspan = create_clusterspan(sorted_clusters, number_of_clusters)
        print('Clusterspan', clusterspan)
        cspd = pd.DataFrame(clusterspan)
        cspd.to_csv('testcsv.csv')
        centers_of_mass = center_of_mass(number_of_clusters, clustered_data) # Kanske ta bort .copy() för bättre prestanda
        
        average_silhouette, cluster_silhouettes = Silhouette(sorted_clusters, number_of_clusters, centers_of_mass, clusterspan)
        print(f'Silhouette: {average_silhouette} \n Cluster silhouettes: {cluster_silhouettes}')
        Sil_mat[index] = MCS, average_silhouette, number_of_clusters
    print(Sil_mat)
    return Sil_mat, sorted_clusters, number_of_clusters, clusterspan, cluster_silhouettes

In [51]:
def cluster_silhouette(clusterdata, CS, cluster_index, CoM):
    # print(CS, CS[cluster_index], CS[cluster_index+1])
    current_cluster = clusterdata[int(CS[cluster_index]):int(CS[cluster_index+1]), :]
    n = np.shape(current_cluster)[0] # Data points in current cluster

    OCS = np.delete(CS, cluster_index+1)
    OCS[cluster_index+1:] -= n
    other_clusters = np.delete(clusterdata.copy(), np.s_[int(CS[cluster_index]):int(CS[cluster_index+1])], axis=0)


    # print(f'Clusterindx: {cluster_index}, OCS: {OCS}, clusterpoints: {n}')
    # other_clusters = np.delete(CoM, cluster_index-1, axis=0)
    # nearest_cluster_index = compare_Euclidean(ref_point, other_clusters, np.shape(other_clusters)[0])[0, 0]

    
    dis_cur_cluster = 0
    NoOC = np.shape(OCS)[0]-1 # NoOC = number of other clusters
    dis_cluster = np.zeros(NoOC)
    dis_near_cluster = 0.0

    # print("NoOC",NoOC)
    for j in range(0, n):
        ref_point = current_cluster[j, :]
        dis_cur_cluster += np.sum(compare_Euclidean(ref_point, current_cluster)[:, 1]) / n
        for l in range(0, NoOC):
            # nearest_cluster_index = int(compare_Euclidean(ref_point, other_clusters, np.shape(other_clusters)[0])[0, 0])
            nearest_cluster = other_clusters[int(OCS[l]):int(OCS[l+1]), :]
            m = np.shape(nearest_cluster)[0] # Data points in nearest cluster
            # print("m", m)
            dis_cluster[l] = np.sum(compare_Euclidean(ref_point, nearest_cluster)[:, 1]) / m
            # if cluster_index == 4:
            #     print(f'NoOC: {NoOC}, dis_cluster: {dis_cluster[l]}, l,n: {l, j}, Compare: {compare_Euclidean(ref_point, nearest_cluster)[:, 1]}') # , cluster {nearest_cluster}')
            # print("j", j, "l",l)
        # print("min", np.min(dis_cluster, axis=0))
        dis_near_cluster += np.min(dis_cluster, axis=0)
        
    # print("near:", dis_near_cluster, "cur:", dis_cur_cluster, "diff:", dis_near_cluster-dis_cur_cluster, "frac:", dis_near_cluster-dis_cur_cluster / dis_near_cluster)


    try:
        average_silhouette = (dis_near_cluster - dis_cur_cluster) / np.maximum(dis_near_cluster, dis_cur_cluster)
        return average_silhouette
    except FloatingPointError:
        print('Current cluster:', current_cluster, 'Dis near:', dis_near_cluster, 'Dis cur:', dis_cur_cluster)
    # print("max", np.maximum(dis_near_cluster, dis_cur_cluster))
    # print("average silhouette", average_silhouette)
    return average_silhouette

Sil_mat, sorted_clusters, number_of_clusters, clusterspan, cluster_silhouettes = calculate_silhouette(norm_data, 
                                                                    start = 2, end = 20, step_size = 2)

MCS: 2
27
Clusterspan [  0.  54.  59.  65.  68.  82.  89.  93.  96.  99. 119. 121. 124. 128.
 131. 135. 138. 142. 149. 152. 156. 165. 168. 173. 177. 180. 192. 194.
 200.]
Silhouette: -0.5649931669290753 
 Cluster silhouettes: [[ 1.00000000e+00 -5.11547464e-01]
 [ 2.00000000e+00 -4.34838999e-01]
 [ 3.00000000e+00 -6.50574539e-01]
 [ 4.00000000e+00  2.02237485e-02]
 [ 5.00000000e+00 -4.77934534e-01]
 [ 6.00000000e+00 -6.83874363e-01]
 [ 7.00000000e+00 -8.13795873e-01]
 [ 8.00000000e+00 -8.38676381e-01]
 [ 9.00000000e+00  5.85933872e-01]
 [ 1.00000000e+01 -8.36322319e-01]
 [ 1.10000000e+01 -7.87420089e-01]
 [ 1.20000000e+01 -7.19390255e-01]
 [ 1.30000000e+01 -8.02063787e-01]
 [ 1.40000000e+01 -8.14407882e-01]
 [ 1.50000000e+01 -8.66936193e-01]
 [ 1.60000000e+01 -7.41218976e-01]
 [ 1.70000000e+01 -4.22945264e-01]
 [ 1.80000000e+01 -8.07468734e-01]
 [ 1.90000000e+01 -7.66323276e-01]
 [ 2.00000000e+01  1.22617966e-01]
 [ 2.10000000e+01 -8.23099474e-01]
 [ 2.20000000e+01 -4.73728442e-01]
 [ 2